In [1]:
import csv
import argparse
import os
import sys
import subprocess
import zipfile
from pathlib import Path
import shutil
import pandas as pd

# Functions

In [2]:
datasets_folder = '/Users/snorre/bin'
def check_datasets_installed():
    """Checks if the 'datasets' command is available in the system PATH."""
    try:
        # Use 'datasets --version' as a simple check
        subprocess.run(['/Users/snorre/bin/datasets', '--version'],
            check=True,  # Will raise a CalledProcessError for non-zero exit codes
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,  # Decode stdout and stderr as text
            capture_output=True,
        )
        print("'datasets' command found.")
        return True
    except FileNotFoundError:
        print("ERROR: 'datasets' command not found. Please install NCBI Datasets CLI:", file=sys.stderr)
        print("See: https://www.ncbi.nlm.nih.gov/datasets/docs/command-line/", file=sys.stderr)
        return False
    except subprocess.CalledProcessError as e:
        print(f"ERROR: 'datasets' command found but returned an error: {e}", file=sys.stderr)
        print("Please ensure your 'datasets' installation is working correctly.", file=sys.stderr)
        return False
    

def download_genome(accession_id, output_dir, include_types="genome,proteome"):
    """
    Downloads genome data for a given accession ID using the 'datasets' tool.

    Args:
        accession_id (str): The NCBI accession ID.
        output_dir (str): The directory where the download zip should be saved.
        include_types (str): Comma-separated string of data types to include
                               (e.g., 'genome,gff3,protein').

    Returns:
        bool: True if download command was executed successfully (exit code 0),
              False otherwise.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Sanitize accession ID for use in filename (replace potentially problematic chars)
    safe_filename_part = "".join(c if c.isalnum() or c in ('_', '-') else '_' for c in accession_id)
    output_filename = os.path.join(output_dir, f"{safe_filename_part}_dataset.zip")

    # Construct the command
    command = [
        '/Users/snorre/bin/datasets',
        'download',
        'genome',
        'accession',
        accession_id,
        '--include', include_types,
        '--filename', output_filename,
        '--no-progressbar' # Optional: cleaner output when running in a loop
    ]

    print(f"Attempting to download: {accession_id} to {output_filename}...")
    print(f"Running command: {' '.join(command)}") # Show the command being run

    try:
        # Execute the command
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        print(f"Successfully downloaded: {accession_id}")
        # print("Output:\n", result.stdout) # Uncomment for detailed output
        return True, output_filename
    except subprocess.CalledProcessError as e:
        print(f"ERROR: Failed to download {accession_id}.", file=sys.stderr)
        print(f"Command failed: {' '.join(e.cmd)}", file=sys.stderr)
        print(f"Return code: {e.returncode}", file=sys.stderr)
        print(f"Error output:\n{e.stderr}", file=sys.stderr)
        # Attempt to remove potentially incomplete zip file
        if os.path.exists(output_filename):
             try:
                 os.remove(output_filename)
                 print(f"Removed potentially incomplete file: {output_filename}", file=sys.stderr)
             except OSError as remove_err:
                 print(f"Warning: Could not remove incomplete file {output_filename}: {remove_err}", file=sys.stderr)
        return False, output_filename
    except FileNotFoundError:
         # This should have been caught by check_datasets_installed, but check again
        print("ERROR: 'datasets' command not found during download attempt.", file=sys.stderr)
        return False, output_filename
    except Exception as e:
        print(f"ERROR: An unexpected error occurred during download of {accession_id}: {e}", file=sys.stderr)
        return False, output_filename
    
def unpack_rename_and_organize(strain, accession_id, downloaded_fn, genomes_folder = '../../data/genomes', proteomes_folder = '../../data/proteomes'):
    extract_directory = Path(downloaded_fn).with_suffix('')

    try:
        with zipfile.ZipFile(downloaded_fn, 'r') as zip_ref:
            zip_ref.extractall(extract_directory)
        print(f"Successfully extracted '{downloaded_fn}' to '{extract_directory}'")

    except FileNotFoundError:
        print(f"Error: Zip file not found at '{downloaded_fn}'")
    except zipfile.BadZipFile:
        print(f"Error: '{downloaded_fn}' is not a valid zip file.")


    # Find the .fna and .faa files in the extracted directory
    fasta_folder = extract_directory / 'ncbi_dataset' / 'data' / accession_id
    fna_file = Path(list(fasta_folder.glob('*.fna'))[0])
    faa_file = Path(list(fasta_folder.glob('*.faa'))[0])
    
    genomes_folder = Path(genomes_folder)
    proteomes_folder = Path(proteomes_folder)
    genomes_folder.mkdir(parents=True, exist_ok=True)
    proteomes_folder.mkdir(parents=True, exist_ok=True)
    fna_file.rename(genomes_folder / f"{strain}_{accession_id}.fna")
    faa_file.rename(proteomes_folder / f"{strain}_{accession_id}.faa")
    print(f"Moved {fna_file} to {genomes_folder / f'{strain}_{accession_id}.fna'}")
    print(f"Moved {faa_file} to {proteomes_folder / f'{strain}_{accession_id}.faa'}")
    # Clean up the temporary folder
    try:
        os.remove(downloaded_fn)
        print(f"Removed temporary file: {downloaded_fn}")
    except OSError as remove_err:
        print(f"Warning: Could not remove temporary file {downloaded_fn}: {remove_err}", file=sys.stderr)
    # Clean up the extracted folder
    try:
        for item in extract_directory.iterdir():
            if item.is_file():
                item.unlink()
            else:
                shutil.rmtree(item)
        extract_directory.rmdir()  # Remove the empty directory
        print(f"Removed temporary directory: {extract_directory}")
    except OSError as remove_err:
        print(f"Warning: Could not remove temporary directory {extract_directory}: {remove_err}", file=sys.stderr)
        
    


In [ ]:
excel_fn = 'strains_to_download.xlsx' # Replace with your actual file path
df = pd.read_excel(excel_fn, sheet_name='Strain Summary', skiprows=1)

In [4]:
for i, row in df.loc[df.Microbiome == 'Metal Working Fluid'].iterrows():
    strain = row['Strain abbreviation']
    
    accession_id = row['Genome accession ID']
    print(strain, accession_id)
    # Download the genome
    success, downloaded_fn = download_genome(accession_id, './temp',  include_types='genome,protein')
    
    if success:
        unpack_rename_and_organize(strain, accession_id, downloaded_fn)
    

1w GCA_040790105.1
Attempting to download: GCA_040790105.1 to ./temp/GCA_040790105_1_dataset.zip...
Running command: /Users/snorre/bin/datasets download genome accession GCA_040790105.1 --include genome,protein --filename ./temp/GCA_040790105_1_dataset.zip --no-progressbar
Successfully downloaded: GCA_040790105.1
Successfully extracted './temp/GCA_040790105_1_dataset.zip' to 'temp/GCA_040790105_1_dataset'
Moved temp/GCA_040790105_1_dataset/ncbi_dataset/data/GCA_040790105.1/GCA_040790105.1_ASM4079010v1_genomic.fna to ../../data/genomes/1w_GCA_040790105.1.fna
Moved temp/GCA_040790105_1_dataset/ncbi_dataset/data/GCA_040790105.1/protein.faa to ../../data/proteomes/1w_GCA_040790105.1.faa
Removed temporary file: ./temp/GCA_040790105_1_dataset.zip
Removed temporary directory: temp/GCA_040790105_1_dataset
At GCA_030505215.1
Attempting to download: GCA_030505215.1 to ./temp/GCA_030505215_1_dataset.zip...
Running command: /Users/snorre/bin/datasets download genome accession GCA_030505215.1 --inc